
## Bamboo XAI Pipeline

 This notebook walks through the **global-to-local XAI layers** for the geopolymer case:

 1. **Global SHAP** on a trained multi-output pipeline (BS, MOE, and EC)
 2. **SHAP clustering** to group mixes by how the model explains them
 3. **TERP** to derive sparse, local surrogates per cluster
 4. **PDPs** for TERP-selected features (1D)
 5. **2D PDP-style trade-off plots** to visualize key trade-offs  

 *** For TERP:  
 Adapted in part from TERP code by the **Tiwary Research Group**
 **Original source:** https://github.com/tiwarylab/TERP/tree/main  
 Licensed under the *MIT License*  
 **For using the TERP method, please cite:**  
 Mehdi, S., & Tiwary, P. (2024). *Thermodynamics-inspired explanations of artificial intelligence*. Nature Communications, 15(1), 7859.


 >Assumes:
 - Tier_smart subset for bamboo was used in this analysis
 - `bamboo_training.csv` with features + [BS, moe, EC]
 - Saved multi-output pipeline (`PIPELINE_PKL`)
 - `cluster_shap`, `run_all_targets`, and `generate_pdp_for_all_targets` modules available


In [1]:
# %%
# 0. Imports & config
import os
import argparse
import json
from sklearn.inspection import PartialDependenceDisplay

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import shap
from xgboost import XGBRegressor  # for XGB models

from clustering import cluster_shap
from TERP_up_final import run_all_targets
from pdp_f1 import generate_pdp_for_all_targets

In [2]:
# Paths to data, model, and SHAP outputs.
# Adjust to match the environment if needed.

# %%
# base paths
FEATURE_CSV  = "bamboo_training.csv" #split training data during model devt (model explanations are based on how model predicts on this dataset subset)
PIPELINE_PKL = r"./models_bamboo/Tier_SMART__MLP.pkl" #tuned model to be explained
SHAP_OUTPUT_DIR = r"./global_shap_outputs/bamboo"

# targets in the multi-output model
targets = ["BS", "moe", "EC"]

RSEED = 0
np.random.seed(RSEED)

### **Load training data & trained pipeline**

- `X_train` – bamboo features + targets
- `pipeline` – scaler + multi-output regressor (BS, MOE, EC)

### MLPProxy definition

 Only needed to load pipelines that were trained using this custom wrapper; does not affect SHAP logic.

In [3]:
# %%
from sklearn.neural_network import MLPRegressor
from sklearn.base import BaseEstimator, RegressorMixin

class MLPProxy(BaseEstimator, RegressorMixin):
    def __init__(self,
                 units1=128, units2=0, units3=0,
                 activation="relu",
                 alpha=1e-4,
                 learning_rate_init=1e-3,
                 batch_size=64,
                 beta_1=0.9, beta_2=0.999,
                 early_stopping=True, validation_fraction=0.15,
                 n_iter_no_change=25, max_iter=1000,
                 solver="adam", shuffle=True,
                 random_state=0):
        self.units1 = units1
        self.units2 = units2
        self.units3 = units3
        self.activation = activation
        self.alpha = alpha
        self.learning_rate_init = learning_rate_init
        self.batch_size = batch_size
        self.beta_1 = beta_1
        self.beta_2 = beta_2
        self.early_stopping = early_stopping
        self.validation_fraction = validation_fraction
        self.n_iter_no_change = n_iter_no_change
        self.max_iter = max_iter
        self.solver = solver
        self.shuffle = shuffle
        self.random_state = random_state

    def _build(self):
        layers = tuple([int(u) for u in (self.units1, self.units2, self.units3) if int(u) > 0])
        if not layers:
            layers = (64,)
        return MLPRegressor(
            hidden_layer_sizes=layers,
            activation=self.activation,
            alpha=self.alpha,
            learning_rate_init=self.learning_rate_init,
            batch_size=self.batch_size,
            beta_1=self.beta_1, beta_2=self.beta_2,
            early_stopping=self.early_stopping,
            validation_fraction=self.validation_fraction,
            n_iter_no_change=self.n_iter_no_change,
            max_iter=self.max_iter,
            solver=self.solver,
            shuffle=self.shuffle,
            random_state=self.random_state
        )

    def fit(self, X, y):
        self._model = self._build()
        return self._model.fit(X, y)

    def predict(self, X):
        return self._model.predict(X)


In [4]:
# %%
X_train = pd.read_csv(FEATURE_CSV)

pipeline = joblib.load(PIPELINE_PKL)
scaler   = pipeline.named_steps["scaler"]
multi_model = pipeline.named_steps["regressor"]

X_scaled = scaler.transform(X_train)
features = X_train.columns.tolist()

os.makedirs(SHAP_OUTPUT_DIR, exist_ok=True)

### 2. Global SHAP (multi-output)
 For each target:
 - Use **TreeExplainer** for tree-based models, else **KernelExplainer**
 - Save:
   - Beeswarm plots (direction + magnitude)
   - Mean |SHAP| bar plots (global ranking)
   - Raw SHAP arrays + mean |SHAP| CSVs


In [6]:
# %%
shap_values_dict = {}
explainer_type_dict = {}

for i, target in enumerate(targets):
    model_i = multi_model.estimators_[i]

    # Choose explainer based on model type
    if hasattr(model_i, "feature_importances_"):
        explainer = shap.TreeExplainer(model_i)
        sv = explainer.shap_values(X_scaled)
        explainer_type = "TreeExplainer"
    else:
        background = shap.sample(X_scaled, 100, random_state=RSEED)
        explainer = shap.KernelExplainer(model_i.predict, background)
        sv = explainer.shap_values(X_scaled, nsamples=100)
        explainer_type = "KernelExplainer"

    explainer_type_dict[target] = explainer_type
    shap_values_dict[target] = sv

    # --- Beeswarm plot ---
    shap.summary_plot(
        sv,
        X_scaled,
        feature_names=features,
        show=False,
        plot_type="dot"
    )
    plt.title(f"SHAP Beeswarm — {target}")
    beeswarm_path = os.path.join(SHAP_OUTPUT_DIR, f"{target}_shap_beeswarm.png")
    plt.savefig(beeswarm_path, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"Saved SHAP beeswarm plot for {target} → {beeswarm_path}")

    # --- Mean |SHAP| bar plot ---
    mean_abs = np.abs(sv).mean(axis=0)
    df_bar = pd.DataFrame({"feature": features, "mean_abs_shap": mean_abs})
    df_bar.sort_values("mean_abs_shap", ascending=True, inplace=True)

    plt.figure(figsize=(6, 5))
    plt.barh(df_bar["feature"], df_bar["mean_abs_shap"])
    plt.title(f"Mean |SHAP| Bar — {target}")
    plt.xlabel("Mean |SHAP value|")
    barplot_path = os.path.join(SHAP_OUTPUT_DIR, f"{target}_shap_barplot.png")
    plt.tight_layout()
    plt.savefig(barplot_path, dpi=200)
    plt.close()
    print(f"Saved SHAP bar plot for {target} → {barplot_path}")

    # --- Save raw SHAP data ---
    np.save(os.path.join(SHAP_OUTPUT_DIR, f"shap_values_{target}.npy"), sv)
    df_bar.to_csv(os.path.join(SHAP_OUTPUT_DIR, f"mean_abs_shap_{target}.csv"), index=False)

# Save combined SHAP arrays + explainer types
joblib.dump(shap_values_dict, os.path.join(SHAP_OUTPUT_DIR, "shap_values.pkl"))
with open(os.path.join(SHAP_OUTPUT_DIR, "explainer_types.json"), "w") as f:
    json.dump(explainer_type_dict, f, indent=2)

print("All SHAP values, bar plots, and summary plots saved.")


  0%|          | 0/71 [00:00<?, ?it/s]

C:\Users\GCOE\AppData\Local\Temp\ipykernel_21892\2507555090.py:23: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(


Saved SHAP beeswarm plot for BS → ./global_shap_outputs/bamboo\BS_shap_beeswarm.png
Saved SHAP bar plot for BS → ./global_shap_outputs/bamboo\BS_shap_barplot.png


  0%|          | 0/71 [00:00<?, ?it/s]

C:\Users\GCOE\AppData\Local\Temp\ipykernel_21892\2507555090.py:23: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(


Saved SHAP beeswarm plot for moe → ./global_shap_outputs/bamboo\moe_shap_beeswarm.png
Saved SHAP bar plot for moe → ./global_shap_outputs/bamboo\moe_shap_barplot.png


  0%|          | 0/71 [00:00<?, ?it/s]

C:\Users\GCOE\AppData\Local\Temp\ipykernel_21892\2507555090.py:23: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(


Saved SHAP beeswarm plot for EC → ./global_shap_outputs/bamboo\EC_shap_beeswarm.png
Saved SHAP bar plot for EC → ./global_shap_outputs/bamboo\EC_shap_barplot.png
All SHAP values, bar plots, and summary plots saved.


### 3. SHAP clustering

 Cluster samples based on their **SHAP profiles** (per target).
 This groups mixes into regimes of similar model behaviour.

In [7]:
from clustering import cluster_shap

target_clusters = cluster_shap(
    joblib.load(os.path.join(SHAP_OUTPUT_DIR, "shap_values.pkl")),
    original_features=X_train.values,
    out_dir="Clusters_Bamboo",
    plot_dir="Cluster_Plots_Bamboo"
)


=== Clustering SHAP for target 'BS' ===
SHAP array shape: (71, 9)


c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


  k=2: silhouette=0.398, inertia=16360.3
  k=3: silhouette=0.296, inertia=12751.7
  k=4: silhouette=0.297, inertia=10353.9
  k=5: silhouette=0.314, inertia=8159.3
  k=6: silhouette=0.294, inertia=7184.0
  k=7: silhouette=0.298, inertia=6559.7


c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The default 

  k=8: silhouette=0.299, inertia=6019.3
  k=9: silhouette=0.265, inertia=5538.5
  k=10: silhouette=0.282, inertia=5078.5
→ Chosen k = 2

→ Saved cluster labels: Clusters_Bamboo\shap_clusters_BS.csv


...

→ Saved cluster labels: Clusters_Bamboo\shap_clusters_moe.csv
→ Saved Silhouette plot: Cluster_Plots_Bamboo\Silhouette_moe.png
→ Saved Elbow plot: Cluster_Plots_Bamboo\Elbow_moe.png
→ Saved PCA data: Clusters_Bamboo\pca_data_moe.csv
→ Saved PCA plot: Cluster_Plots_Bamboo\PCA_SHAP_moe.png
→ Cluster 0: 30 pts, saved pure→'Clusters_Bamboo\moe_cluster_0_pure.npy' and aug→'Clusters_Bamboo\moe_cluster_0_augmented.npy'
→ Cluster 1: 41 pts, saved pure→'Clusters_Bamboo\moe_cluster_1_pure.npy' and aug→'Clusters_Bamboo\moe_cluster_1_augmented.npy'

=== Clustering SHAP for target 'EC' ===
SHAP array shape: (71, 9)
  k=2: silhouette=0.391, inertia=4.0
  k=3: silhouette=0.388, inertia=2.9
  k=4: silhouette=0.343, inertia=2.2
  k=5: silhouette=0.345, inertia=1.7


c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The default 

  k=6: silhouette=0.335, inertia=1.5
  k=7: silhouette=0.332, inertia=1.2
  k=8: silhouette=0.321, inertia=1.1
  k=9: silhouette=0.323, inertia=1.0
  k=10: silhouette=0.321, inertia=0.9
→ Chosen k = 2



c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The default 

→ Saved cluster labels: Clusters_Bamboo\shap_clusters_EC.csv
→ Saved Silhouette plot: Cluster_Plots_Bamboo\Silhouette_EC.png
→ Saved Elbow plot: Cluster_Plots_Bamboo\Elbow_EC.png
→ Saved PCA data: Clusters_Bamboo\pca_data_EC.csv
→ Saved PCA plot: Cluster_Plots_Bamboo\PCA_SHAP_EC.png
→ Cluster 0: 31 pts, saved pure→'Clusters_Bamboo\EC_cluster_0_pure.npy' and aug→'Clusters_Bamboo\EC_cluster_0_augmented.npy'
→ Cluster 1: 40 pts, saved pure→'Clusters_Bamboo\EC_cluster_1_pure.npy' and aug→'Clusters_Bamboo\EC_cluster_1_augmented.npy'

Finished clustering all targets.


### 4. TERP on each SHAP cluster
 Run TERP across all bamboo targets and SHAP clusters.
 This will:
 - Generate local neighbourhoods
 - Fit sparse surrogates
 - Store optimal feature sets and weights

In [8]:
args = argparse.Namespace(
    targets       = ["BS", "moe", "EC"],
    pipeline_path = PIPELINE_PKL,
    cluster_dir   = "Clusters_Bamboo",
    output_dir    = "TERP_Results_Bamboo",
    feature_csv   = FEATURE_CSV,
    seed          = 0,
    num_samples   = 3000,
    cutoff        = 5
)

all_optimal_indices = run_all_targets(args)
print("Collected optimal indices:", all_optimal_indices)

c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\G

Collected optimal indices: {'BS': {'pure': {'0': 'TERP_Results_Bamboo\\pure\\optimal_feature_indices_BS_cluster_0_pure.npy', '1': 'TERP_Results_Bamboo\\pure\\optimal_feature_indices_BS_cluster_1_pure.npy'}, 'augmented': {'0': 'TERP_Results_Bamboo\\augmented\\optimal_feature_indices_BS_cluster_0_augmented.npy', '1': 'TERP_Results_Bamboo\\augmented\\optimal_feature_indices_BS_cluster_1_augmented.npy'}}, 'moe': {'pure': {'0': 'TERP_Results_Bamboo\\pure\\optimal_feature_indices_moe_cluster_0_pure.npy', '1': 'TERP_Results_Bamboo\\pure\\optimal_feature_indices_moe_cluster_1_pure.npy'}, 'augmented': {'0': 'TERP_Results_Bamboo\\augmented\\optimal_feature_indices_moe_cluster_0_augmented.npy', '1': 'TERP_Results_Bamboo\\augmented\\optimal_feature_indices_moe_cluster_1_augmented.npy'}}, 'EC': {'pure': {'0': 'TERP_Results_Bamboo\\pure\\optimal_feature_indices_EC_cluster_0_pure.npy', '1': 'TERP_Results_Bamboo\\pure\\optimal_feature_indices_EC_cluster_1_pure.npy'}, 'augmented': {'0': 'TERP_Results_B

c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


### 5. 1D PDPs for TERP-selected features

 For each **bamboo target × cluster**:
 - Use TERP-selected features as candidates
 - Plot 1D PDPs for **cluster medoids/centroids**
 - Save plots under `PDP_Plots_Bamboo`

In [9]:
model_pipeline = joblib.load(PIPELINE_PKL)
X_train        = pd.read_csv(FEATURE_CSV)
regression_targets = ["BS", "moe", "EC"]
feature_names  = [c for c in X_train.columns if c not in regression_targets]

generate_pdp_for_all_targets(
    targets               = ["BS", "moe", "EC"],
    pipeline              = model_pipeline,
    X_train               = X_train,
    feature_names         = feature_names,
    cluster_dir           = "Clusters_Bamboo",
    optimal_indices_dirs  = ["TERP_Results_Bamboo/augmented", "TERP_Results_Bamboo/pure"],
    pdp_out_dir           = "PDP_Plots_Bamboo"
)


Generating PDPs for target: BS
Looking in: TERP_Results_Bamboo/augmented\optimal_feature_indices_BS_cluster_0_augmented.npy
  Cluster 0 (augmented) → features [0 1 3 5]
BS_cluster_0_augmented.npy → shape (36, 9)


c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\G

Looking in: TERP_Results_Bamboo/augmented\optimal_feature_indices_BS_cluster_0_pure.npy
Looking in: TERP_Results_Bamboo/pure\optimal_feature_indices_BS_cluster_0_pure.npy
  Cluster 0 (pure) → features [0 3 5]
BS_cluster_0_pure.npy → shape (35, 9)


c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\G

Looking in: TERP_Results_Bamboo/augmented\optimal_feature_indices_BS_cluster_1_augmented.npy
  Cluster 1 (augmented) → features [0 1 3 5]
BS_cluster_1_augmented.npy → shape (37, 9)


...

Looking in: TERP_Results_Bamboo/augmented\optimal_feature_indices_EC_cluster_1_augmented.npy
  Cluster 1 (augmented) → features [2 4]
EC_cluster_1_augmented.npy → shape (41, 9)


c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\G

Looking in: TERP_Results_Bamboo/augmented\optimal_feature_indices_EC_cluster_1_pure.npy
Looking in: TERP_Results_Bamboo/pure\optimal_feature_indices_EC_cluster_1_pure.npy
  Cluster 1 (pure) → features [2 4]
EC_cluster_1_pure.npy → shape (40, 9)


c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\GCOE\Downloads\THESIS_tools\.venv\lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\G

Completed PDPs for all clusters of target: EC


### 6. 2D PDPs

 To keep things lightweight, we only:
 - Use **sklearn PartialDependenceDisplay** for 2D surfaces
 - Plot over the full bamboo dataset (no medoids, no sample scatter)

 You can select a few **key pairs** (e.g., geometry vs mass, density vs thickness)
 that correspond to the TERP-selected drivers for BS, MOE, or EC.


In [10]:
def generate_2d_pdp_for_cluster(
    pipeline,
    X,                      # full feature DataFrame (no targets)
    feature_names,
    target_name,            # BS", "moe", "EC"
    target_index,           # index of target in multi-output model
    feature_pairs,          # list of (i, j) indices in feature_names
    cluster_assignments,    # dict: target -> array of labels
    cluster_label,          # which cluster to use (int)
    out_dir="PDP_Plots_Bamboo_2D_cluster",
    percentile_clip=(5, 95) # global range clip for consistency
):
    """
    Cluster-based 2D PDP: averages over samples in a single SHAP cluster,
    BUT x/y axis ranges are taken from the FULL dataset X so that
    plots for the same feature pair are directly comparable.

    - Uses only rows where cluster_assignments[target_name] == cluster_label
      to compute the PDP surface.
    - Uses percentiles of the full X for axis limits so different clusters
      / targets share the same ranges for a given feature pair.
    """
    os.makedirs(out_dir, exist_ok=True)

    labels = cluster_assignments[target_name]
    mask   = (labels == cluster_label)
    if not mask.any():
        print(f"[WARN] No samples for {target_name} cluster {cluster_label}")
        return

    X_cluster = X.loc[mask]
    print(f"{target_name} cluster {cluster_label}: {mask.sum()} samples used for PDP")

    for (i, j) in feature_pairs:
        feat_i = feature_names[i]
        feat_j = feature_names[j]

        # --- GLOBAL (dataset-wide) ranges for consistency across plots ---
        low, high = percentile_clip
        x_min, x_max = np.percentile(X[feat_i], [low, high])
        y_min, y_max = np.percentile(X[feat_j], [low, high])

        fig, ax = plt.subplots(figsize=(6, 5))
        try:
            # PDP averaged over the cluster samples only
            PartialDependenceDisplay.from_estimator(
                pipeline,
                X_cluster,               # cluster = what you condition on
                features=[(i, j)],
                target=target_index,
                grid_resolution=50,
                ax=ax,
                feature_names=feature_names
            )

            # enforce the SAME axes for any plot using this feature pair
            ax.set_xlim(x_min, x_max)
            ax.set_ylim(y_min, y_max)

            ax.set_title(
                f"2D PDP — {target_name} (cluster {cluster_label})\n"
                f"{feat_i} vs {feat_j}",
                fontsize=11
            )
            fig.tight_layout()
            fname = f"PDP2D_{target_name}_C{cluster_label}_{feat_i}_{feat_j}.png"
            fig.savefig(os.path.join(out_dir, fname), dpi=150, bbox_inches="tight")
            plt.close(fig)
            print(f"Saved cluster 2D PDP: {fname}")
        except Exception as e:
            plt.close(fig)
            print(
                f"Failed 2D PDP for {feat_i} vs {feat_j} "
                f"({target_name}, cluster {cluster_label}): {e}"
            )


##### generating the selected plots

In [ ]:
cluster_dir = "Clusters_Bamboo"   # or "Clusters" – keep consistent with cluster_shap(out_dir=...)

# Targets for BAMBOO, not GPC
regression_targets = ["BS", "moe", "EC"]

def load_cluster_labels_from_csv(cluster_label_dir, targets):
    assignments = {}
    for t in targets:
        path = os.path.join(cluster_label_dir, f"shap_clusters_{t}.csv")
        df = pd.read_csv(path)
        assignments[t] = df["cluster"].to_numpy()
    return assignments

cluster_assignments_dict = load_cluster_labels_from_csv(cluster_dir, regression_targets)

# Features only (no targets)
X_features_only = X_train[[c for c in X_train.columns if c not in regression_targets]]
feature_names   = X_features_only.columns.tolist()

In [12]:
# === choose cluster IDs that matter ===
bs_cluster_high = 1   # e.g. high-BS regime
ec_cluster_low  = 0   # e.g. low-EC regime

# === choose feature pairs based on your bamboo feature names ===
# example only – adjust to whatever you actually want to show:
bs_pairs = [
    (feature_names.index("rho"), feature_names.index("mom")),   # density vs bending moment
]

ec_pairs = [
    (feature_names.index("rho"), feature_names.index("m")),     # density vs mass
]

# --- 2D PDP for BS high-strength cluster ---
generate_2d_pdp_for_cluster(
    pipeline            = model_pipeline,   # MLP bamboo pipeline you want to explain
    X                   = X_features_only,
    feature_names       = feature_names,
    target_name         = "BS",            # <-- bamboo target
    target_index        = 0,               # 0: BS, 1: moe, 2: EC  (order from your MLP multi-output)
    feature_pairs       = bs_pairs,
    cluster_assignments = cluster_assignments_dict,
    cluster_label       = bs_cluster_high,
    out_dir             = "PDP_Plots_Bamboo_2D_cluster"
)

# --- 2D PDP for EC low-carbon cluster ---
generate_2d_pdp_for_cluster(
    pipeline            = model_pipeline,
    X                   = X_features_only,
    feature_names       = feature_names,
    target_name         = "EC",
    target_index        = 2,
    feature_pairs       = ec_pairs,
    cluster_assignments = cluster_assignments_dict,
    cluster_label       = ec_cluster_low,
    out_dir             = "PDP_Plots_Bamboo_2D_cluster"
)


BS cluster 1: 36 samples used for PDP
Saved cluster 2D PDP: PDP2D_BS_C1_rho_mom.png
EC cluster 0: 31 samples used for PDP
Saved cluster 2D PDP: PDP2D_EC_C0_rho_m.png
